In [ ]:
import pandas as pd

ruta = "/content/drive/MyDrive/datos DEIS/mortalidad_2005_2023.parquet"

df = pd.read_parquet(ruta)

In [ ]:
df.columns

Index(['PROVRES', 'SEXO', 'CAUSA', 'MAT', 'GRUPEDAD', 'CUENTA', 'anio',
       'source_file'],
      dtype='object')

In [ ]:
codes_path = "/content/drive/MyDrive/datos DEIS/Descripción de los archivos de defunciones.xlsx"  # pegá la ruta real

import pandas as pd

xls = pd.ExcelFile(codes_path)
print(xls.sheet_names)

['DISEÑO', 'PROVRES', 'SEXO', 'CODMUER', 'MAT']


In [ ]:
import pandas as pd

codes_path = "/content/drive/MyDrive/datos DEIS/Descripción de los archivos de defunciones.xlsx"

prov = pd.read_excel(codes_path, sheet_name="PROVRES", dtype=str)

print("Shape:", prov.shape)
print("Columnas:", prov.columns.tolist())
prov.head(10)

Shape: (26, 2)
Columnas: ['CODIGO', 'VALOR']


,CODIGO,VALOR
0,02,Ciudad Aut. de Buenos Aires
1,06,Buenos Aires
2,10,Catamarca
3,14,Córdoba
4,18,Corrientes
5,22,Chaco
6,26,Chubut
7,30,Entre Ríos
8,34,Formosa
9,38,Jujuy


In [ ]:
prov["CODIGO"].unique()
prov.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   CODIGO  26 non-null     object
 1   VALOR   26 non-null     object
dtypes: object(2)
memory usage: 548.0+ bytes


In [ ]:
prov_ref = prov.copy()

prov_ref.columns = ["codigo", "provincia"]
prov_ref["codigo"] = prov_ref["codigo"].str.strip()
prov_ref["provincia"] = prov_ref["provincia"].str.strip()

# Asegurar 2 dígitos (02, 06, 10...) por seguridad
prov_ref["codigo"] = prov_ref["codigo"].str.zfill(2)

prov_ref.head(10)

,codigo,provincia
0,02,Ciudad Aut. de Buenos Aires
1,06,Buenos Aires
2,10,Catamarca
3,14,Córdoba
4,18,Corrientes
5,22,Chaco
6,26,Chubut
7,30,Entre Ríos
8,34,Formosa
9,38,Jujuy


In [ ]:
# Cargar el parquet unificado (si todavía no lo tenés en variable df)
df = pd.read_parquet("/content/drive/MyDrive/datos DEIS/mortalidad_2005_2023.parquet")

# Copia analítica
df_analitico = df.copy()

# Normalizar PROVRES para matchear (sin cambiar el original)
df_analitico["PROVRES_norm"] = df_analitico["PROVRES"].astype(str).str.strip().str.zfill(2)

# Merge
df_analitico = df_analitico.merge(
    prov_ref,
    left_on="PROVRES_norm",
    right_on="codigo",
    how="left"
)

# Chequeos
print("Filas antes:", df.shape[0])
print("Filas después:", df_analitico.shape[0])
print("Provincias nulas tras merge:", df_analitico["provincia"].isna().sum())

df_analitico[["PROVRES", "PROVRES_norm", "provincia"]].head(10)

Filas antes: 922900
Filas después: 922900
Provincias nulas tras merge: 0


,PROVRES,PROVRES_norm,provincia
0,50,50,Mendoza
1,14,14,Córdoba
2,62,62,Río Negro
3,74,74,San Luis
4,6,06,Buenos Aires
5,46,46,La Rioja
6,6,06,Buenos Aires
7,22,22,Chaco
8,26,26,Chubut
9,26,26,Chubut


In [ ]:
sexo = pd.read_excel(codes_path, sheet_name="SEXO", dtype=str)

print("Shape:", sexo.shape)
print("Columnas:", sexo.columns.tolist())
sexo.head()

Shape: (3, 2)
Columnas: ['CODIGO', 'VALOR']


,CODIGO,VALOR
0,1,Varón
1,2,Mujer
2,9,Sin especificar


In [ ]:
sexo_ref = sexo.copy()

sexo_ref.columns = ["codigo", "sexo_desc"]
sexo_ref["codigo"] = sexo_ref["codigo"].str.strip()
sexo_ref["sexo_desc"] = sexo_ref["sexo_desc"].str.strip()

sexo_ref.head()

,codigo,sexo_desc
0,1,Varón
1,2,Mujer
2,9,Sin especificar


In [ ]:
# Normalizar SEXO para matchear
df_analitico["SEXO_norm"] = df_analitico["SEXO"].astype(str).str.strip()

# Merge
df_analitico = df_analitico.merge(
    sexo_ref,
    left_on="SEXO_norm",
    right_on="codigo",
    how="left"
)

# Chequeos
print("Filas:", df_analitico.shape[0])
print("Sexo nulo tras merge:", df_analitico["sexo_desc"].isna().sum())

df_analitico[["SEXO", "SEXO_norm", "sexo_desc"]].head(10)


Filas: 922900
Sexo nulo tras merge: 0


,SEXO,SEXO_norm,sexo_desc
0,2,2,Mujer
1,1,1,Varón
2,1,1,Varón
3,2,2,Mujer
4,1,1,Varón
5,2,2,Mujer
6,1,1,Varón
7,2,2,Mujer
8,2,2,Mujer
9,1,1,Varón


In [ ]:
mat = pd.read_excel(codes_path, sheet_name="MAT", dtype=str)

print("Shape:", mat.shape)
print("Columnas:", mat.columns.tolist())
mat.head(10)

Shape: (4, 2)
Columnas: ['CODIGO', 'VALOR']


,CODIGO,VALOR
0,M,Muerte materna
1,T,Muerte materna tardía
2,S,Secuela de causa obstétrica
3,Nulo,No es muerte materna


In [ ]:
df_analitico["MAT"].value_counts(dropna=False)

,count
MAT,
None,861858
,56601
M,3975
T,463
S,3


In [ ]:
# Ver distribución por año
df_analitico.groupby("anio")["MAT"].apply(
    lambda x: x.value_counts(dropna=False)
)

anio     
2005         45324
      M        229
      T         14
2006  NaN    45375
      M        264
      T         11
2007  NaN    46798
      M        240
      T         22
2008  NaN    46715
      M        244
      T         21
2009  NaN    47313
      M        282
      T         18
2010  NaN    47474
      M        238
      T         23
2011  NaN    46864
      M        231
      T         15
2012  NaN    47155
      M        204
      T         23
2013  NaN    48249
      M        197
      T         25
2014  NaN    48105
      M        222
      T         19
2015  NaN    40973
             11277
      M        234
      T         27
2016  NaN    50951
      M        199
      T         27
2017  NaN    49630
      M        167
      T         31
      S          3
2018  NaN    49602
      M        200
      T         29
2019  NaN    49635
      M        158
      T         27
2020  NaN    48617
      M        173
      T         27
2021  NaN    49414
      M        217
      T         34
2022  NaN    50000
      M        147
      T         41
2023  NaN    48988
      M        129
      T         29
Name: MAT, dtype: int64

In [ ]:
# 1️⃣ Cuántos registros son
df_analitico[
    (df_analitico["anio"] == "2015") &
    (df_analitico["MAT"] == "11277")
].shape

(0, 14)

In [ ]:
# 2️⃣ Ver algunas filas completas
df_analitico[
    (df_analitico["anio"] == "2015") &
    (df_analitico["MAT"] == "11277")
].head(10)

,PROVRES,SEXO,CAUSA,MAT,GRUPEDAD,CUENTA,anio,source_file,PROVRES_norm,codigo_x,provincia,SEXO_norm,codigo_y,sexo_desc


In [ ]:
# 3️⃣ Ver columnas alrededor para detectar corrimiento
df_analitico[
    (df_analitico["anio"] == "2015") &
    (df_analitico["MAT"] == "11277")
][["PROVRES","SEXO","CAUSA","MAT","GRUPEDAD","CUENTA"]].head(10)

,PROVRES,SEXO,CAUSA,MAT,GRUPEDAD,CUENTA


In [ ]:
mat_vc = df_analitico["MAT"].value_counts(dropna=False)
mat_vc

,count
MAT,
None,861858
,56601
M,3975
T,463
S,3


In [ ]:
valid = {"M","T","S",""}
mask_valid = df_analitico["MAT"].isna() | df_analitico["MAT"].isin(valid)

df_analitico.loc[~mask_valid, ["anio","MAT"]].value_counts().head(20)

,,count
anio,MAT,
2005,,45324
2015,,11277


In [ ]:
df_analitico.loc[~mask_valid, ["anio","MAT","PROVRES","SEXO","CAUSA","GRUPEDAD","CUENTA"]].head(10)

,anio,MAT,PROVRES,SEXO,CAUSA,GRUPEDAD,CUENTA
0,2005,,50,2,R99,11_50 a 54,10
1,2005,,14,1,F10,12_55 a 59,1
2,2005,,62,1,N05,17_80 y más,1
3,2005,,74,2,C26,17_80 y más,3
4,2005,,6,1,G30,13_60 a 64,3
5,2005,,46,2,J96,15_70 a 74,2
6,2005,,6,1,V49,13_60 a 64,31
7,2005,,22,2,N17,17_80 y más,5
8,2005,,26,2,E14,17_80 y más,8
9,2005,,26,1,J18,13_60 a 64,3


In [ ]:
# Crear columna descriptiva sin tocar MAT original

df_analitico["MAT_desc"] = df_analitico["MAT"]

# Reemplazar valores vacíos o NaN
df_analitico.loc[
    df_analitico["MAT_desc"].isna() |
    (df_analitico["MAT_desc"].str.strip() == ""),
    "MAT_desc"
] = "N"

# Crear diccionario manual basado en hoja MAT
mat_dict = {
    "M": "Muerte materna",
    "T": "Muerte materna tardía",
    "S": "Secuela de causa obstétrica",
    "N": "No es muerte materna"
}

df_analitico["MAT_desc"] = df_analitico["MAT_desc"].map(mat_dict)

df_analitico["MAT_desc"].value_counts()

,count
MAT_desc,
No es muerte materna,918459
Muerte materna,3975
Muerte materna tardía,463
Secuela de causa obstétrica,3


In [ ]:
codmuer = pd.read_excel(codes_path, sheet_name="CODMUER", dtype=str)

print("Shape:", codmuer.shape)
print("Columnas:", codmuer.columns.tolist())
codmuer.head(10)

Shape: (2053, 2)
Columnas: ['CODIGO', 'VALOR']


,CODIGO,VALOR
0,A00,Cólera
1,A01,Fiebres tifoidea y paratifoidea
2,A02,Otras infecciones debidas a Salmonella
3,A03,Shigelosis
4,A04,Otras infecciones intestinales bacterianas
5,A05,Otras intoxicaciones alimentarias bacterianas ...
6,A06,Amebiasis
7,A07,Otras enfermedades intestinales debidas a prot...
8,A08,Infecciones intestinales debidas a virus y otr...
9,A09,Otras gastroenteritis y colitis de origen infe...


In [ ]:
print("Ejemplos de CAUSA en dataset:")
df_analitico["CAUSA"].unique()[:20]

Ejemplos de CAUSA en dataset:


array(['R99', 'F10', 'N05', 'C26', 'G30', 'J96', 'V49', 'N17', 'E14',
       'J18', 'C34', 'Q00', 'C64', 'I35', 'C79', 'M32', 'G00', 'I64',
       'W14', 'I77'], dtype=object)

In [ ]:
print("Cantidad de códigos únicos en dataset:")
df_analitico["CAUSA"].nunique()

Cantidad de códigos únicos en dataset:


1509

In [ ]:
# Normalizar diccionario
codmuer_ref = codmuer.copy()
codmuer_ref.columns = ["codigo", "causa_desc"]
codmuer_ref["codigo"] = codmuer_ref["codigo"].str.strip()

# Conjunto de códigos
codigos_dataset = set(df_analitico["CAUSA"].unique())
codigos_diccionario = set(codmuer_ref["codigo"].unique())

# Diferencia
codigos_no_mapeados = codigos_dataset - codigos_diccionario

print("Cantidad de códigos en dataset:", len(codigos_dataset))
print("Cantidad de códigos en diccionario:", len(codigos_diccionario))
print("Códigos no mapeados:", len(codigos_no_mapeados))

list(codigos_no_mapeados)[:20]

Cantidad de códigos en dataset: 1509
Cantidad de códigos en diccionario: 2050
Códigos no mapeados: 9


['k80', 'R97', 'U10', 'j18', 'I84', 'u07', 'U12', 'k74', 'b99']

In [ ]:
df_analitico["CAUSA_norm"] = df_analitico["CAUSA"].str.strip().str.upper()

# Recalcular diferencia
codigos_dataset = set(df_analitico["CAUSA_norm"].unique())
codigos_diccionario = set(codmuer_ref["codigo"].unique())

codigos_no_mapeados = codigos_dataset - codigos_diccionario

print("Códigos no mapeados tras normalizar:", len(codigos_no_mapeados))
list(codigos_no_mapeados)



Códigos no mapeados tras normalizar: 4


['U12', 'U10', 'I84', 'R97']

In [ ]:
# Ver si en el diccionario existen códigos que empiecen con I84
[c for c in codmuer_ref["codigo"] if c.startswith("I84")]

# Ver si existen códigos que empiecen con R97
[c for c in codmuer_ref["codigo"] if c.startswith("R97")]

# Ver si existen códigos que empiecen con U10
[c for c in codmuer_ref["codigo"] if c.startswith("U10")]

# Ver si existen códigos que empiecen con U12
[c for c in codmuer_ref["codigo"] if c.startswith("U12")]

[]

In [ ]:
# Tabla ref CODMUER
codmuer_ref = codmuer.copy()
codmuer_ref.columns = ["codigo", "causa_desc"]
codmuer_ref["codigo"] = codmuer_ref["codigo"].str.strip().str.upper()
codmuer_ref["causa_desc"] = codmuer_ref["causa_desc"].str.strip()

# Normalizamos causa en dataset (sin tocar CAUSA original)
df_analitico["CAUSA_norm"] = df_analitico["CAUSA"].astype(str).str.strip().str.upper()

# Merge
df_analitico = df_analitico.merge(
    codmuer_ref,
    left_on="CAUSA_norm",
    right_on="codigo",
    how="left"
)

# Fallback para no mapeados
df_analitico["causa_desc"] = df_analitico["causa_desc"].fillna("No mapeado (fuera de diccionario)")

print("No mapeados:", (df_analitico["causa_desc"] == "No mapeado (fuera de diccionario)").sum())
df_analitico[["CAUSA","CAUSA_norm","causa_desc"]].head(12)

No mapeados: 2459


,CAUSA,CAUSA_norm,causa_desc
0,R99,R99,Otras causas mal definidas y las no especifica...
1,F10,F10,Trastornos mentales y del comportamiento debid...
2,N05,N05,Síndrome nefrítico no especificado
3,C26,C26,Tumor maligno de otros sitios y de los mal def...
4,G30,G30,Enfermedad de Alzheimer
5,J96,J96,Insuficiencia respiratoria no clasificada en o...
6,V49,V49,Ocupante de automóvil lesionado en otros accid...
7,N17,N17,Insuficiencia renal aguda
8,E14,E14,Diabetes mellitus no especificada
9,J18,J18,Neumonía organismo no especificado


In [ ]:
df_analitico[df_analitico["causa_desc"] == "No mapeado (fuera de diccionario)"]["CAUSA_norm"].value_counts()

,count
CAUSA_norm,
U07,2424
I84,15
R97,10
U12,5
U10,5


In [ ]:
# Crear copia enriquecida del diccionario
codmuer_enriquecido = codmuer_ref.copy()

# Agregar U07 manualmente
codmuer_enriquecido = pd.concat([
    codmuer_enriquecido,
    pd.DataFrame([{
        "codigo": "U07",
        "causa_desc": "COVID-19"
    }])
], ignore_index=True)

# Volver a hacer el merge limpio
df_analitico = df_analitico.drop(columns=["codigo", "causa_desc"], errors="ignore")

df_analitico = df_analitico.merge(
    codmuer_enriquecido,
    left_on="CAUSA_norm",
    right_on="codigo",
    how="left"
)

df_analitico["causa_desc"] = df_analitico["causa_desc"].fillna(
    "No mapeado (fuera de diccionario CODMUER)"
)

# Verificar U07
df_analitico[df_analitico["CAUSA_norm"] == "U07"]["causa_desc"].value_counts()

,count
causa_desc,
No mapeado (fuera de diccionario CODMUER),2424
COVID-19,2424


In [ ]:
	# 1️⃣ Eliminar columnas de merge anteriores si existen
df_analitico = df_analitico.drop(columns=["codigo", "causa_desc"], errors="ignore")

# 2️⃣ Crear diccionario enriquecido final
codmuer_enriquecido = codmuer_ref.copy()

codmuer_enriquecido = pd.concat([
    codmuer_enriquecido,
    pd.DataFrame([{
        "codigo": "U07",
        "causa_desc": "COVID-19"
    }])
], ignore_index=True)

# 3️⃣ Merge limpio
df_analitico = df_analitico.merge(
    codmuer_enriquecido,
    left_on="CAUSA_norm",
    right_on="codigo",
    how="left"
)

# 4️⃣ Fallback
df_analitico["causa_desc"] = df_analitico["causa_desc"].fillna(
    "No mapeado (fuera de diccionario CODMUER)"
)

# 5️⃣ Verificar solo U07
df_analitico[df_analitico["CAUSA_norm"] == "U07"]["causa_desc"].value_counts()

,count
causa_desc,
No mapeado (fuera de diccionario CODMUER),4848
COVID-19,4848


In [ ]:
codmuer_enriquecido[codmuer_enriquecido["codigo"] == "U07"]

codmuer_enriquecido = codmuer_enriquecido.drop_duplicates(subset=["codigo"], keep="last")
codmuer_enriquecido[codmuer_enriquecido["codigo"] == "U07"]





,codigo,causa_desc
2053,U07,COVID-19


In [ ]:
# BORRAR solo las columnas del merge de causa, no todo el df
df_analitico = df_analitico.drop(columns=["codigo", "causa_desc"], errors="ignore")

# Merge limpio con diccionario deduplicado
df_analitico = df_analitico.merge(
    codmuer_enriquecido,
    left_on="CAUSA_norm",
    right_on="codigo",
    how="left",
    validate="m:1"  # <- esto hace que pandas falle si el diccionario no es 1 por codigo
)

df_analitico["causa_desc"] = df_analitico["causa_desc"].fillna("No mapeado (fuera de diccionario CODMUER)")

# Chequeo puntual U07
df_analitico[df_analitico["CAUSA_norm"] == "U07"]["causa_desc"].value_counts()



,count
causa_desc,
COVID-19,9696


In [ ]:
df_analitico.shape

(930172, 18)

In [ ]:
df_raw = pd.read_parquet("/content/drive/MyDrive/datos DEIS/mortalidad_2005_2023.parquet")
print(df_raw.shape)
df_raw.head(3)

(922900, 8)


,PROVRES,SEXO,CAUSA,MAT,GRUPEDAD,CUENTA,anio,source_file
0,50,2,R99,,11_50 a 54,10,2005,Mortalidad año 2005.xlsx
1,14,1,F10,,12_55 a 59,1,2005,Mortalidad año 2005.xlsx
2,62,1,N05,,17_80 y más,1,2005,Mortalidad año 2005.xlsx


In [ ]:
df_raw.shape

(922900, 8)

In [ ]:
import pandas as pd

# --- 0) Partimos del crudo ---
df_analitico_clean = df_raw.copy()

# --- 1) PROVRES -> provincia ---
prov = pd.read_excel(codes_path, sheet_name="PROVRES", dtype=str)
prov_ref = prov.copy()
prov_ref.columns = ["codigo", "provincia"]
prov_ref["codigo"] = prov_ref["codigo"].str.strip().str.zfill(2)
prov_ref["provincia"] = prov_ref["provincia"].str.strip()

df_analitico_clean["PROVRES_norm"] = df_analitico_clean["PROVRES"].astype(str).str.strip().str.zfill(2)

df_analitico_clean = df_analitico_clean.merge(
    prov_ref,
    left_on="PROVRES_norm",
    right_on="codigo",
    how="left",
    validate="m:1"
)

# --- 2) SEXO -> sexo_desc ---
sexo = pd.read_excel(codes_path, sheet_name="SEXO", dtype=str)
sexo_ref = sexo.copy()
sexo_ref.columns = ["codigo_sexo", "sexo_desc"]
sexo_ref["codigo_sexo"] = sexo_ref["codigo_sexo"].str.strip()
sexo_ref["sexo_desc"] = sexo_ref["sexo_desc"].str.strip()

df_analitico_clean["SEXO_norm"] = df_analitico_clean["SEXO"].astype(str).str.strip()

df_analitico_clean = df_analitico_clean.merge(
    sexo_ref,
    left_on="SEXO_norm",
    right_on="codigo_sexo",
    how="left",
    validate="m:1"
)

# --- 3) MAT -> MAT_desc (sin tocar MAT original) ---
mat_dict = {
    "M": "Muerte materna",
    "T": "Muerte materna tardía",
    "S": "Secuela de causa obstétrica",
    "N": "No es muerte materna"
}

df_analitico_clean["MAT_desc"] = df_analitico_clean["MAT"]

# N (No es muerte materna) para NaN o vacío
df_analitico_clean.loc[
    df_analitico_clean["MAT_desc"].isna() |
    (df_analitico_clean["MAT_desc"].astype(str).str.strip() == ""),
    "MAT_desc"
] = "N"

# Mapear a descripción
df_analitico_clean["MAT_desc"] = df_analitico_clean["MAT_desc"].map(mat_dict)

# --- 4) CAUSA (CIE10) -> causa_desc (con U07 enriquecido) ---
codmuer = pd.read_excel(codes_path, sheet_name="CODMUER", dtype=str)
codmuer_ref = codmuer.copy()
codmuer_ref.columns = ["codigo_causa", "causa_desc"]
codmuer_ref["codigo_causa"] = codmuer_ref["codigo_causa"].str.strip().str.upper()
codmuer_ref["causa_desc"] = codmuer_ref["causa_desc"].str.strip()

# Enriquecer SOLO U07
codmuer_enriquecido = pd.concat([
    codmuer_ref,
    pd.DataFrame([{"codigo_causa": "U07", "causa_desc": "COVID-19"}])
], ignore_index=True).drop_duplicates(subset=["codigo_causa"], keep="last")

df_analitico_clean["CAUSA_norm"] = df_analitico_clean["CAUSA"].astype(str).str.strip().str.upper()

df_analitico_clean = df_analitico_clean.merge(
    codmuer_enriquecido,
    left_on="CAUSA_norm",
    right_on="codigo_causa",
    how="left",
    validate="m:1"
)

df_analitico_clean["causa_desc"] = df_analitico_clean["causa_desc"].fillna(
    "No mapeado (fuera de diccionario CODMUER)"
)

# --- 5) Chequeos de integridad ---
print("SHAPE FINAL:", df_analitico_clean.shape)
print("U07 ->", df_analitico_clean[df_analitico_clean["CAUSA_norm"]=="U07"]["causa_desc"].value_counts())
print("\nNO MAPEADOS (top):")
print(df_analitico_clean[df_analitico_clean["causa_desc"]=="No mapeado (fuera de diccionario CODMUER)"]["CAUSA_norm"].value_counts().head(10))
print("\nMAT_desc:")
print(df_analitico_clean["MAT_desc"].value_counts())

SHAPE FINAL: (922900, 18)
U07 -> causa_desc
COVID-19    2424
Name: count, dtype: int64

NO MAPEADOS (top):
CAUSA_norm
I84    15
R97    10
U12     5
U10     5
Name: count, dtype: int64

MAT_desc:
MAT_desc
No es muerte materna           918459
Muerte materna                   3975
Muerte materna tardía             463
Secuela de causa obstétrica         3
Name: count, dtype: int64


In [ ]:
out_parquet = "/content/drive/MyDrive/datos DEIS/mortalidad_2005_2023_analitico.parquet"
df_analitico_clean.to_parquet(out_parquet, index=False)
print("Guardado:", out_parquet)

Guardado: /content/drive/MyDrive/datos DEIS/mortalidad_2005_2023_analitico.parquet


In [ ]:
out_csv = "/content/drive/MyDrive/datos DEIS/mortalidad_2005_2023_analitico.csv"
df_analitico_clean.to_csv(out_csv, index=False, encoding="utf-8")
print("Guardado:", out_csv)

Guardado: /content/drive/MyDrive/datos DEIS/mortalidad_2005_2023_analitico.csv


DECISIÓN METODOLÓGICA — MAT (interpretación de nulos)
En MAT, valores nulos (NaN) y cadenas vacías representan la categoría conceptual “Nulo” del diccionario (“No es muerte materna”). Se creó MAT_desc sin modificar MAT original.

DECISIÓN METODOLÓGICA — CAUSA (CIE-10) y diccionario CODMUER
Se creó CAUSA_norm (mayúsculas) para evitar discrepancias por formato. Se mapeó CAUSA_norm usando la hoja CODMUER. Se incorporó manualmente solo U07 → COVID-19 por relevancia (2424 registros). Los códigos restantes fuera de diccionario (I84, R97, U10, U12) se etiquetan como “No mapeado (fuera de diccionario CODMUER)” sin alterar los códigos originales.

In [ ]:
import os
print(os.path.exists(out_parquet), os.path.exists(out_csv))

True True
